# Multi-Modal Neural Network for Construction Cost Prediction

This notebook builds a model combining:
- Tabular data (MLP)
- Sentinel-2 imagery (CNN)
- VIIRS imagery (CNN)

We fuse all modalities into a final regression model.

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader

import pandas as pd
from pathlib import Path
import tifffile as tiff

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## Dataset handling
Loading and preparing dataset for the model

In [2]:
DataPath = Path("..") / "Processed data"
ImgPath = DataPath / "processed_composite"

train_df = pd.read_csv(DataPath / "processed_data.csv")
philippines_df = pd.read_csv(DataPath / "processed_philippines.csv")
japan_df = pd.read_csv(DataPath / "processed_japan.csv")

print(f"Train shape: {train_df.shape}")
print(f"Japan shape: {japan_df.shape}")
print(f"Philippines shape: {philippines_df.shape}")
train_df.head()

Train shape: (1024, 21)
Japan shape: (567, 18)
Philippines shape: (457, 20)


,data_id,geolocation_name,quarter_label,country,year,deflated_gdp_usd,us_cpi,landlocked,region_economic_classification,access_to_airport,...,access_to_highway,access_to_railway,straight_distance_to_capital_km,seismic_hazard_zone,flood_risk_class,tropical_cyclone_wind_risk,tornadoes_wind_risk,koppen_climate_zone,construction_cost_per_m2_usd,processed_imgs
0,LP81L,0,3,0,0.0,0.002133,0.059541,0,1,0,...,1,0,0.496774,2,1,3,0,0,129.997420,dinagat_islands_2019-Q3.pt
1,E7EOB,1,2,1,1.0,0.919424,0.973574,1,3,0,...,1,1,0.238710,2,1,1,0,3,1567.878774,29000_nara_2024-Q2.pt
2,WAOUA,2,1,1,0.2,0.954862,0.085467,0,3,1,...,1,1,0.290323,2,1,1,0,5,2009.827701,05000_akita_2020-Q1.pt
3,2IZ5P,3,4,0,0.2,0.000000,0.119109,1,1,1,...,1,0,0.561290,3,1,2,0,0,377.279961,cotabato_2020-Q4.pt
4,RJ5XF,4,3,0,0.0,0.002133,0.059541,1,1,1,...,1,1,0.041935,2,1,2,0,1,163.905688,pampanga_2019-Q3.pt


In [3]:
class MultiModalDataset(Dataset):
    def __init__(self, df, numeric_cols, categorical_cols,
                 imgs_col, target_col):
        self.df = df.reset_index(drop=True)
        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols
        self.imgs_col = imgs_col
        self.target_col = target_col
        
    def __len__(self):
        return len(self.df)
    
    def load_imgs(self, path):
        img = torch.load(ImgPath / path, weights_only=True)
        return img
    
    def __getitem__(self, idx):

        x_num = torch.tensor(
            self.df.loc[idx, self.numeric_cols].values.astype("float32"),
            dtype=torch.float32
        )

        x_cat = torch.tensor(
            self.df.loc[idx, self.categorical_cols].values.astype("int64"),
            dtype=torch.long
        )

        imgs = self.load_imgs(self.df.loc[idx, self.imgs_col])
        img_s = imgs['sentinel']
        img_v = imgs['viirs']

        y = torch.tensor(
            float(self.df.loc[idx, self.target_col]),
            dtype=torch.float32
        )

        return {
            "numeric": x_num,
            "categorical": x_cat,
            "sentinel_img": img_s,
            "viirs_img": img_v,
            "target": y
        }

In [4]:
numeric_cols = [
    'year', 'deflated_gdp_usd', 'us_cpi',
    'straight_distance_to_capital_km'
]

target_col = 'construction_cost_per_m2_usd'

categorical_cols = [
    'access_to_airport', 'access_to_highway', 'access_to_port',
    'access_to_railway', 'country',
    'flood_risk_class', 'geolocation_name', 'koppen_climate_zone',
    'landlocked', 'quarter_label', 'region_economic_classification',
    'seismic_hazard_zone', 'tornadoes_wind_risk',
    'tropical_cyclone_wind_risk'
]

imgs_col = "processed_imgs"

In [5]:
#Convert the columns to the appropriate data types

for col in numeric_cols:
    train_df[col] = pd.to_numeric(train_df[col], errors='coerce')
for col in categorical_cols:
    train_df[col] = train_df[col].astype('category').cat.codes + 1  # Start categories from 1, reserve 0 for padding

print(train_df[numeric_cols].dtypes)
print(train_df[categorical_cols].dtypes)

year                               float64
deflated_gdp_usd                   float64
us_cpi                             float64
straight_distance_to_capital_km    float64
dtype: object
access_to_airport                 int8
access_to_highway                 int8
access_to_port                    int8
access_to_railway                 int8
country                           int8
flood_risk_class                  int8
geolocation_name                  int8
koppen_climate_zone               int8
landlocked                        int8
quarter_label                     int8
region_economic_classification    int8
seismic_hazard_zone               int8
tornadoes_wind_risk               int8
tropical_cyclone_wind_risk        int8
dtype: object


## Tabular Model (MLP with Embeddings)

In [6]:
class TabularModel(nn.Module):
    def __init__(self, num_numeric, cat_dims, emb_dims):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(cat_dim, emb_dim)
            for cat_dim, emb_dim in zip(cat_dims, emb_dims)
        ])

        emb_total_dim = sum(emb_dims)

        self.mlp = nn.Sequential(
            nn.Linear(num_numeric + emb_total_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )

    def forward(self, x_numeric, x_categorical):
        emb = [emb_layer(x_categorical[:, i]) 
               for i, emb_layer in enumerate(self.embeddings)]
        emb = torch.cat(emb, dim=1)

        x = torch.cat([x_numeric, emb], dim=1)
        return self.mlp(x)

## CNN Image Encoder (ResNet Backbone)

In [7]:
class ImageEncoder(nn.Module):
    def __init__(self, in_channels=3, output_dim=128):
        super().__init__()
        self.backbone = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, output_dim)

    def forward(self, x):
        return self.backbone(x)

## Fusion Model

In [8]:
class FusionModel(nn.Module):
    def __init__(self, tabular_model, sentinel_model, viirs_model):
        super().__init__()

        self.tabular = tabular_model
        self.sentinel = sentinel_model
        self.viirs = viirs_model

        self.head = nn.Sequential(
            nn.Linear(64 + 128 + 128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, x_num, x_cat, img_sentinel, img_viirs):
        t_feat = self.tabular(x_num, x_cat)
        s_feat = self.sentinel(img_sentinel)
        v_feat = self.viirs(img_viirs)

        x = torch.cat([t_feat, s_feat, v_feat], dim=1)
        return self.head(x)

## Loss Function

In [9]:
criterion = nn.SmoothL1Loss()

## Example Model Initialization

In [10]:
num_numeric = len(numeric_cols)
cat_cols = categorical_cols
cat_dims = [train_df[col].nunique() + 1 for col in cat_cols]
emb_dims = [min(50, (dim + 1) // 2) for dim in cat_dims]

tabular_model = TabularModel(num_numeric, cat_dims, emb_dims)
sentinel_model = ImageEncoder(in_channels=12, output_dim=128)
viirs_model = ImageEncoder(in_channels=1,  output_dim=128)

base_model = FusionModel(tabular_model, sentinel_model, viirs_model)

base_model.to(device)

FusionModel(
  (tabular): TabularModel(
    (embeddings): ModuleList(
      (0-5): 6 x Embedding(3, 2)
      (6): Embedding(126, 50)
      (7): Embedding(8, 4)
      (8): Embedding(3, 2)
      (9-10): 2 x Embedding(5, 3)
      (11): Embedding(4, 2)
      (12): Embedding(3, 2)
      (13): Embedding(5, 3)
    )
    (mlp): Sequential(
      (0): Linear(in_features=85, out_features=128, bias=True)
      (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Linear(in_features=128, out_features=64, bias=True)
      (4): ReLU()
    )
  )
  (sentinel): ImageEncoder(
    (backbone): ResNet(
      (conv1): Conv2d(12, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0)

## Training Step (Skeleton)

In [11]:
def train_step(model, optimizer, batch):
    model.train()
    x_num = batch["numeric"].to(device)
    x_cat = batch["categorical"].to(device)
    img_s = batch["sentinel_img"].to(device)
    img_v = batch["viirs_img"].to(device)
    y = batch["target"].to(device)

    # Check inputs
    assert not torch.isnan(x_num).any(), "NaN in x_num"
    assert not torch.isnan(img_s).any(), "NaN in sentinel image"
    assert not torch.isnan(img_v).any(), "NaN in viirs image"
    assert not torch.isnan(y).any(), "NaN in target"

    preds = model(x_num, x_cat, img_s, img_v)
    assert not torch.isnan(preds).any(), "NaN in predictions"

    loss = criterion(preds.squeeze(), y)
    assert not torch.isnan(loss), "NaN in loss"

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    return loss.item()

## Training the model

In [12]:
import datetime


dataset = MultiModalDataset(train_df, numeric_cols, categorical_cols, imgs_col, target_col)

train_loader = DataLoader(dataset, batch_size=64, shuffle=True)

def train_model(model : torch.nn.Module, loader : DataLoader, epochs=10):
    
    losses = []
    n_batch = len(loader)

    print(f'{datetime.datetime.now().time()}  |  Starting training...')
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(1, epochs + 1):
        total_loss = 0

        for batch in loader:
            loss = train_step(model, optimizer, batch)
            total_loss += loss
        
        torch.cuda.empty_cache()
        avg_loss = total_loss / n_batch
        losses.append(avg_loss)

        print('{}  |  Epoch {}  |  Training loss {:.3f}'.format(datetime.datetime.now().time(), epoch, avg_loss))
    
    return losses


losses = train_model(base_model, train_loader, epochs=10)

ModelPath = Path("..") / "Models"

torch.save({
    'model': base_model.state_dict(),
    'losses': losses
}, ModelPath / "fusion_model.pth")

16:23:31.517851  |  Starting training...
16:23:45.012546  |  Epoch 1  |  Training loss 1078.654
16:23:49.615309  |  Epoch 2  |  Training loss 888.814
16:23:54.146093  |  Epoch 3  |  Training loss 486.224
16:23:58.937218  |  Epoch 4  |  Training loss 262.067
16:24:03.958803  |  Epoch 5  |  Training loss 199.287
16:24:08.548610  |  Epoch 6  |  Training loss 180.280
16:24:13.099153  |  Epoch 7  |  Training loss 184.336
16:24:17.668745  |  Epoch 8  |  Training loss 185.619
16:24:22.146473  |  Epoch 9  |  Training loss 154.372
16:24:26.636492  |  Epoch 10  |  Training loss 170.311
